In [1]:
# Import packages

import os

%matplotlib qt
from matplotlib.pyplot import plot, show, title

import scipy
import scipy.signal

import numpy
from numpy import loadtxt
from numpy import savetxt
import numpy as np

import biosppy
from biosppy.signals import ecg

from opensignalsreader import OpenSignalsReader

import pyhrv.tools as tools
from pyhrv.hrv import hrv

import csv
from csv import reader

In [22]:
# Creating the final .csv file with all the information

hrv_csv_name = '/home/eduarda/Desktop/HRVdata/ProcessedECGData.csv'

#with open(hrv_csv_name, 'a', encoding='UTF8', newline='') as f:
#    header = ['file', 'label', 'rr_mean', 'hr_mean', 'sdnn', 'rmssd', 'sdann', 'pnn50']
#    writer = csv.writer(f)
    # write the header
#    writer.writerow(header)

# Functions

## Splitting ECG in 30s time frames

In [3]:
def split_ECG(time_frame, time, samples):
    
    batch = time_frame // time #number of samples that equal 30s

    number_of_arrays = samples // batch # number of total splits of ecg file

    split_values = np.array_split(filtered_ecg, number_of_arrays)
    
    return batch, number_of_arrays, split_values

## Detecting R Peaks with Biospy (Hamilton)

https://biosppy.readthedocs.io/en/stable/biosppy.signals.html#biosppy-signals-ecg

In [4]:
def rPeakHamiltonTempo(filtered_ecg, fs, time):
    #Returns R-peak location indices (sample number, IT IS NOT TIME)
    rPeakLocation = biosppy.signals.ecg.hamilton_segmenter(signal=filtered_ecg, sampling_rate=fs)
    
    #Obtaining R-peak values in time (s): R-peak sample number -> R-peak time value
    tt = []
    
    for i in rPeakLocation:
        xx = i * time
        tt.append(xx)
        i = i + 1
        
    return tt

## Detecting R Peaks with default parameters

https://biosppy.readthedocs.io/en/stable/biosppy.signals.html#biosppy-signals-ecg

In [5]:
def rPeak(filtered_ecg, fs, time):
    #Returns R-peak location indices (sample number, IT IS NOT TIME)
    rPeakLocation = biosppy.signals.ecg.ecg(filtered_ecg, sampling_rate=fs, show = False)[2]
    
    #Obtaining R-peak values in time (s): R-peak sample number -> R-peak time value
    tt = []
    
    for i in rPeakLocation:
        xx = i * time
        tt.append(xx)
        i = i + 1
    
    return tt

## Calculating all rpeaks in the 30s time-frames

The variable `rPeakss` returns the time in which the R-peaks occured within a 30s time frame
The list `rPeaks` contains all rPeaks of the ecg

In [6]:
#l is a list that contains x numbers of lists, x =number_of_arrays. 
#l[0] contains the time in which the rPeaks for the first 30s of measurmente occured.


def rPeaks_final(split_values, fs, time):
    rPeaks=[]
    for n in range (len(split_values)):
        #R-peak Hamilton
        #rPeakss = rPeakHamiltonTempo(split_values[n], fs, time)

        #R-peaks default parameters
        rPeakss = rPeak(split_values[n], fs, time)


        rPeaks.append(rPeakss)
    return rPeaks

## Calculating RR intervals (RR)

In [7]:
def detectrr(rPeaks):
    
    #Obtaining RR intervals (difference in time between consecutive R-Peaks)
    rr = []
    for i in range(len(rPeaks) - 1):
        xx = rPeaks[i + 1] - rPeaks[i]
        rr.append(xx)
        i = i + 1
    return rr

# ECG file upload -> Processing -> HRV Calculation -> Save data

fs - Recording ECG frequency (Swell - 2048 Hz; Wesad - 700 Hz)

ecg_duration - duration of the ECG in seconds

label - 0 is baseline, 1 is stress

time_frame - analysing data in 30s time-frame

https://pyhrv.readthedocs.io/en/latest/_pages/api/hrv.html

In [21]:
# Path
path = "/media/eduarda/Seagate/datasets/"

# open file in read 'r' mode
with open('/media/eduarda/Seagate/time_duration.csv', 'r') as read_obj:
    csv_reader = reader(read_obj)
    
    for row in csv_reader:
        
        file = row[0]
        ecg_duration1 = row[1]
        ecg_duration = float(ecg_duration1)
        
        label1 = row[2]
        label = int(label1)
        
        fs1 = row[3]
        fs = int(fs1)
        
        # Join various path components
        final_path = os.path.join(path, file)
        
        print("Reading the file: ", file)
        
        filtered_ecg = loadtxt(final_path, delimiter=',')
        
        
        samples = sum(1 for row in filtered_ecg) # total number of data points in the ecg file
        time = ecg_duration / samples # time between consecutive data points in the .csv file
        time_frame = 30

        # Spliting the ECG in 30s
        batch, number_of_arrays, split_values = split_ECG(time_frame, time, samples)

        #Rpeaks for each 30s split
        rPeaks = rPeaks_final(split_values, fs, time)


        #HRV calculation and save .csv file
        x=len(split_values)


        #stops the hrv function from plotting all the graphics
        import matplotlib.pyplot as plt
        plt.ioff()

        for n in range (x):

            #Calculating HRV based on rPeaks calculated before
            hrv1 = hrv(rpeaks=(rPeaks[n]), sampling_rate=fs, show= False, plot_tachogram= False, plot_ecg= False)

            #Calculating HRV with filtered ecg, default parameters
            #hrv1 = hrv(signal=(split_values[n]), sampling_rate=fs,  show= False, plot_tachogram= False, plot_ecg= False)


            #Calculating RR interval
            rr_interval = detectrr(rPeaks[n])

            t=0
            for g in range (len(rr_interval)):
                t = rr_interval[g] + t
                g = g + 1
            rr_mean = t / (len(rr_interval))


            # the variable `hrv` is classbiosppy.utils.ReturnTuple(values, names=None), must be converted to ordered dictionary
            hrv_dic = hrv1.as_dict()
            hr_mean = hrv_dic["hr_mean"]
            sdnn = hrv_dic["sdnn"]
            rmssd = hrv_dic["rmssd"]
            sdann = hrv_dic["sdann"]
            pnn50 = hrv_dic["pnn50"]

            #col1 = (n + 1) * time_frame

            data = [file, label, rr_mean , hr_mean, sdnn, rmssd, sdann, pnn50]

            #open file in append 'a' mode
            with open(hrv_csv_name, 'a', encoding='UTF8', newline='') as f:
                writer = csv.writer(f)
                # write the data
                writer.writerow(data)

            del hrv1 #cleaning the variable
        

Reading the file:  S2_baseline_ECG_values.csv
[0.6939338061938062, 1.8087782817182816, 2.921726763236763, 3.9740034365634362, 4.954232337662337, 5.898437352647353, 6.799034505494505, 7.718591598401598, 8.579372877122877, 9.421194215784215, 10.285767482517482, 11.154132737262737, 12.013018021978022, 12.868111318681319, 13.734580579420578, 14.591569870129868, 15.501646993006991, 16.46670794205794, 17.48864871128871, 18.466981618381617, 19.447210519480517, 20.39899951048951, 21.31286862137862, 22.185025864135863, 23.04959913086913, 23.93123634365634, 24.85268943056943, 25.73811863136863, 26.610275874125872, 27.47864112887113, 28.453182047952048, 29.607842397602397]
[0.4019507292707292, 1.4200995104895104, 2.3927444355644356, 3.31798951048951, 4.197730729270729, 5.031968091908092, 5.86430946053946, 6.63597902097902, 7.511928251748251, 8.653316643356643, 9.779537082917082, 10.826125774225774, 12.02628998001998, 13.26247807192807, 14.267354895104894, 15.183119999999999, 16.129221008991006, 1

[0.7546056143856144, 1.643826803196803, 2.54631995004995, 3.572052707292707, 4.660353266733266, 5.858621478521478, 7.0796416183816175, 8.194486093906093, 9.373794365634366, 10.553102637362636, 11.603483316683317, 12.74107972027972, 13.85782018981019, 15.004896563436562, 16.070445194805195, 17.172017712287712, 18.374077912087913, 19.369474765234763, 20.647374725274723, 21.779283146853146, 22.81828786213786, 23.908484415584415, 25.110544615384615, 26.261412977022975, 27.440721248751245, 28.669325364635363]
[0.659805914085914, 1.8258422277722277, 2.798487152847153, 3.803363976023976, 4.925792427572428, 6.057700849150849, 7.223737162837162, 8.476989200799201, 9.737825214785214, 10.869733636363636, 12.003538051948052, 13.160094395604395, 14.216163056943056, 15.317735574425573, 16.50273182817183, 17.680144105894104, 18.8841002997003, 20.124080379620377, 21.329932567432564, 22.41633713286713, 23.538765584415582, 24.685841958041955, 25.796894445554443, 26.854859100899098, 28.077775234765234, 2

[0.5460462737262737, 1.5148992107892107, 2.4761681718281716, 3.534132827172827, 4.739985014985015, 5.885165394605394, 7.108081528471528, 8.330997662337662, 9.457218101898102, 10.627046403596403, 11.834794585414585, 12.945847072927073, 13.998123746253745, 15.065568371628371, 16.06475721278721, 16.98241831168831, 17.972127182817182, 18.986483976023976, 20.000840769230766, 21.007613586413584, 22.05609827172827, 23.13112687312687, 24.187195534465534, 25.256536153846152, 26.227285084915085, 27.209409980019977, 28.23703873126873, 29.31585932067932]
[0.963164955044955, 2.1026573526473524, 3.255421708291708, 4.385434135864136, 5.409270899100899, 6.486195494505494, 7.608623946053946, 8.731052397602397, 9.775745094905094, 10.89058957042957, 12.024393986013985, 13.146822437562436, 14.195307122877121, 15.175536023976022, 16.193684805194803, 17.264921418581416, 18.343742007992006, 19.32965889110889, 20.37435158841159, 21.445588201798202, 22.549056713286713, 23.517909650349647, 24.44125873126873, 25

/home/eduarda/anaconda3/lib/python3.7/site-packages/pyhrv/utils.py:368: UserWarning: Signal duration is to short for segmentation into 300000s. Input data will be returned.
  warnings.warn("Signal duration is to short for segmentation into %is. Input data will be returned." % duration)
/home/eduarda/anaconda3/lib/python3.7/site-packages/pyhrv/time_domain.py:303: UserWarning: Signal duration too short for SDANN computation.
  warnings.warn("Signal duration too short for SDANN computation.")
/home/eduarda/anaconda3/lib/python3.7/site-packages/pyhrv/time_domain.py:576: UserWarning: CAUTION: The TINN computation is currently providing incorrect results in the most cases due to a malfunction of the function. This function will be reviewed over the next updates to solve this issue
  warnings.warn('CAUTION: The TINN computation is currently providing incorrect results in the most cases due to a '
/home/eduarda/anaconda3/lib/python3.7/site-packages/pyhrv/nonlinear.py:286: UserWarning: Not enou

/home/eduarda/anaconda3/lib/python3.7/site-packages/pyhrv/time_domain.py:576: UserWarning: CAUTION: The TINN computation is currently providing incorrect results in the most cases due to a malfunction of the function. This function will be reviewed over the next updates to solve this issue
  warnings.warn('CAUTION: The TINN computation is currently providing incorrect results in the most cases due to a '
/home/eduarda/anaconda3/lib/python3.7/site-packages/pyhrv/nonlinear.py:286: UserWarning: Not enough NNI samples for Detrended Fluctuations Analysis.
  warnings.warn("Not enough NNI samples for Detrended Fluctuations Analysis.")
/home/eduarda/anaconda3/lib/python3.7/site-packages/pyhrv/utils.py:368: UserWarning: Signal duration is to short for segmentation into 300000s. Input data will be returned.
  warnings.warn("Signal duration is to short for segmentation into %is. Input data will be returned." % duration)
/home/eduarda/anaconda3/lib/python3.7/site-packages/pyhrv/time_domain.py:303:

/home/eduarda/anaconda3/lib/python3.7/site-packages/pyhrv/nonlinear.py:286: UserWarning: Not enough NNI samples for Detrended Fluctuations Analysis.
  warnings.warn("Not enough NNI samples for Detrended Fluctuations Analysis.")
/home/eduarda/anaconda3/lib/python3.7/site-packages/pyhrv/utils.py:368: UserWarning: Signal duration is to short for segmentation into 300000s. Input data will be returned.
  warnings.warn("Signal duration is to short for segmentation into %is. Input data will be returned." % duration)
/home/eduarda/anaconda3/lib/python3.7/site-packages/pyhrv/time_domain.py:303: UserWarning: Signal duration too short for SDANN computation.
  warnings.warn("Signal duration too short for SDANN computation.")
/home/eduarda/anaconda3/lib/python3.7/site-packages/pyhrv/time_domain.py:576: UserWarning: CAUTION: The TINN computation is currently providing incorrect results in the most cases due to a malfunction of the function. This function will be reviewed over the next updates to sol

... (4 outputs left out)

Reading the file:  S3_baseline_ECG_values.csv
[0.3457493734335839, 1.2762661654135337, 2.2924686716791975, 3.2530506265664156, 4.0963784461152875, 4.870556390977443, 5.613165914786967, 6.461003508771929, 7.371977944862154, 8.555042105263157, 9.732093233082706, 10.886595488721802, 12.03207819548872, 13.124946867167917, 14.323043609022555, 15.588786967418544, 16.65910676691729, 17.721910275689222, 18.811772431077692, 20.05947669172932, 21.38234385964912, 22.706714285714284, 23.76500802005012, 24.877419047619043, 25.914667167919795, 26.9128305764411, 27.906484210526312, 29.03994085213032, 30.20797243107769]
[0.566728320802005, 1.5964601503759397, 2.8291318295739343, 4.153502255639097, 5.401206516290726, 6.54067619047619, 7.690668671679197, 8.831641604010024, 10.040261152882204, 11.289468671679197, 12.42893834586466, 13.63755789473684, 14.901797994987467, 16.190090225563907, 17.347598997493733, 18.556218546365912, 19.81143909774436, 21.002019548872177, 22.01671879699248, 22.972790977443605

[1.0853523809523808, 2.2924686716791975, 3.562721804511278, 4.858530325814535, 5.8987849624060145, 6.88492230576441, 7.788380451127819, 8.6271984962406, 9.529153383458645, 11.122607017543858, 12.4725328320802, 13.631544862155387, 14.957418546365913, 16.281788972431077, 17.54452581453634, 18.61785213032581, 19.76934786967418, 21.020058646616537, 22.28730526315789, 23.377167418546364, 24.485068671679194, 25.662119799498743, 26.951915288220547, 28.052300250626562, 29.31654035087719]
[1.082345864661654, 2.212795989974937, 3.4409578947368415, 4.673629573934837, 5.919830576441102, 6.981130827067668, 8.102561403508771, 9.287128822055136, 10.512284210526314, 11.590120300751877, 12.756648621553882, 14.010365914786966, 15.229508270676689, 16.307344360902253, 17.374657644110272, 18.43144812030075, 19.572421052631576, 20.572087719298242, 21.573257644110274, 22.702204511278193, 23.889778446115287, 24.922516791979948, 26.063489724310774, 27.287141854636587, 28.524323308270674, 29.58562355889724]
[1.

/home/eduarda/anaconda3/lib/python3.7/site-packages/pyhrv/utils.py:368: UserWarning: Signal duration is to short for segmentation into 300000s. Input data will be returned.
  warnings.warn("Signal duration is to short for segmentation into %is. Input data will be returned." % duration)
/home/eduarda/anaconda3/lib/python3.7/site-packages/pyhrv/time_domain.py:303: UserWarning: Signal duration too short for SDANN computation.
  warnings.warn("Signal duration too short for SDANN computation.")
/home/eduarda/anaconda3/lib/python3.7/site-packages/pyhrv/time_domain.py:576: UserWarning: CAUTION: The TINN computation is currently providing incorrect results in the most cases due to a malfunction of the function. This function will be reviewed over the next updates to solve this issue
  warnings.warn('CAUTION: The TINN computation is currently providing incorrect results in the most cases due to a '
/home/eduarda/anaconda3/lib/python3.7/site-packages/pyhrv/nonlinear.py:286: UserWarning: Not enou

KeyboardInterrupt: 